# Quant Signal Project

پروژه ساده برای تولید سیگنال خرید/فروش روی بورس تهران.

**مراحل:**
1. نصب کتابخانه‌ها
2. دریافت داده یک نماد
3. دریافت لیست کل بازار
4. فیلتر درشت
5. محاسبه فیچرها
6. امتیازدهی
7. خروجی جدول رنگی

## ! pip install pytse-client requests

In [4]:
import sys
import re
import time
import requests
import pytse_client as tse

print("✅ ایمپورت‌ها انجام شد")

✅ ایمپورت‌ها انجام شد


In [5]:
_cache = {}


def _num(value, kind=float, default=0):
    """تبدیل امن به عدد."""
    if value is None:
        return default
    try:
        return kind(value)
    except (TypeError, ValueError):
        return default


def _str(value, default="—"):
    """تبدیل امن به متن."""
    if value is None:
        return default
    return str(value).strip()


def _day(value):
    text = str(value).split(".")[0]
    return text[:10] if len(text) >= 10 and text[4] == "-" else text


def load(symbol):
    """داده یک نماد رو از tsetmc می‌گیره."""
    if symbol in _cache:
        return _cache[symbol]

    t = tse.Ticker(symbol)

    history = [
        {
            "date": _day(r["date"]),
            "adj": _num(r["adjClose"], int),
            "last": _num(r["close"], int),
            "volume": _num(r["volume"], int),
        }
        for r in t.history.to_dict("records")
    ]

    clients = [
        {
            "date": _day(r["date"]),
            "real": _num(r["individual_buy_vol"], int),
            "legal": _num(r["corporate_buy_vol"], int),
        }
        for r in t.client_types.to_dict("records")
    ]
    clients.reverse()

    data = {
        "title": _str(t.title),
        "group": _str(t.group_name),
        "market": _str(t.flow),
        "last": _num(t.last_price, int),
        "yday": _num(t.yesterday_price, int),
        "pe": _num(t.p_e_ratio, float),
        "history": history,
        "clients": clients,
    }
    _cache[symbol] = data
    return data


print("✅ تابع load آماده است")

✅ تابع load آماده است


In [6]:
t = time.time()
d = load("فولاد")
print(f"⏱ زمان: {time.time()-t:.1f} ثانیه")
print(f"نام: {d['title']}")
print(f"گروه: {d['group']}")
print(f"بازار: {d['market']}")
print(f"آخرین: {d['last']}")
print(f"تعداد روز تاریخچه: {len(d['history'])}")
print(f"تعداد روز حقیقی/حقوقی: {len(d['clients'])}")

⏱ زمان: 5.7 ثانیه
نام: فولاد مبارکه اصفهان
گروه: فلزات اساسي
بازار: بورس
آخرین: 4410
تعداد روز تاریخچه: 4247
تعداد روز حقیقی/حقوقی: 3863


In [7]:
MARKET_URL = "https://cdn.tsetmc.com/api/ClosingPrice/GetMarketWatch"


def get_market_overview(min_volume=100_000):
    """کل بازار رو با یه درخواست می‌گیره."""
    resp = requests.get(
        MARKET_URL,
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()

    out = []
    for item in data.get("marketWatch", []):
        symbol = item.get("lva") or ""
        last = item.get("pdv") or 0
        close = item.get("pcl") or 0
        yday = item.get("py") or 0
        volume = item.get("qtj") or 0

        if not symbol or last == 0 or volume < min_volume:
            continue

        out.append({
            "symbol": symbol,
            "last": int(last),
            "close": int(close),
            "yday": int(yday),
            "volume": int(volume),
        })

    return out


print("✅ تابع get_market_overview آماده است")

✅ تابع get_market_overview آماده است


In [8]:
market = get_market_overview()
print(f"✅ تعداد نمادهای فعال: {len(market)}")
print(f"\n۵ تای اول:")
for m in market[:5]:
    print(f"  {m['symbol']:15s} | last={m['last']:>10,} | vol={m['volume']:>15,}")

✅ تعداد نمادهای فعال: 0

۵ تای اول:


In [9]:
def shortlist(candidates, top_n=15):
    """فیلتر درشت: نمادهای مثبت امروز، مرتب بر اساس حجم."""
    positive = []
    for c in candidates:
        if c["yday"] > 0 and c["last"] > c["yday"]:
            positive.append(c)

    positive.sort(key=lambda c: c["volume"], reverse=True)
    return positive[:top_n]


print("✅ تابع shortlist آماده است")

✅ تابع shortlist آماده است


In [10]:
short = shortlist(market, top_n=15)
print(f"✅ کاندیداها: {len(short)}")
print(f"\nلیست:")
for i, c in enumerate(short, 1):
    print(f"  {i:2d}. {c['symbol']:15s} (حجم: {c['volume']:,})")

✅ کاندیداها: 0

لیست:


In [11]:
def feature_return_1d(data):
    """آیا بازده امروز مثبته؟"""
    if data["yday"] == 0:
        return False
    return (data["last"] - data["yday"]) / data["yday"] > 0


def feature_return_5d(data):
    """آیا بازده ۵ روزه مثبته؟"""
    h = data["history"]
    if len(h) < 6 or h[-6]["adj"] == 0:
        return False
    return h[-1]["adj"] > h[-6]["adj"]


def feature_real_buyer(data):
    """آیا حقیقی بیشتر از حقوقی خریده؟"""
    if not data["clients"]:
        return False
    last = data["clients"][-1]
    return last["real"] > last["legal"]


def feature_volume_spike(data, factor=1.5):
    """آیا حجم امروز از میانگین ۵ روز قبل بیشتره؟"""
    h = data["history"]
    if len(h) < 6:
        return False
    avg5 = sum(d["volume"] for d in h[-6:-1]) / 5
    if avg5 == 0:
        return False
    return h[-1]["volume"] > avg5 * factor


print("✅ چهار فیچر آماده است")

✅ چهار فیچر آماده است


In [12]:
def evaluate(data):
    """چهار فیچر رو حساب می‌کنه و (feats, score, signal) برمی‌گردونه."""
    feats = {
        "R1": feature_return_1d(data),
        "R5": feature_return_5d(data),
        "RB": feature_real_buyer(data),
        "V":  feature_volume_spike(data),
    }
    score = sum(feats.values())

    if score == 4:
        sig = "BUY"
    elif score == 3:
        sig = "WATCH"
    elif score == 2:
        sig = "WAIT"
    else:
        sig = "SELL"

    return feats, score, sig


print("✅ تابع evaluate آماده است")

✅ تابع evaluate آماده است


In [13]:
feats, score, sig = evaluate(d)
print(f"فولاد: {score}/4 → {sig}")
print(f"فیچرها: {feats}")

فولاد: 1/4 → SELL
فیچرها: {'R1': False, 'R5': True, 'RB': False, 'V': False}


In [14]:
class C:
    RESET = "\033[0m"
    BOLD = "\033[1m"
    RED = "\033[31m"
    GREEN = "\033[32m"
    YELLOW = "\033[33m"
    CYAN = "\033[36m"
    MAGENTA = "\033[35m"
    GRAY = "\033[90m"


# اگه تو Jupyter هستیم (نه ترمینال)، رنگ‌ها رو خالی می‌کنیم
if not sys.stdout.isatty():
    C.RESET = C.BOLD = C.RED = C.GREEN = C.YELLOW = ""
    C.CYAN = C.MAGENTA = C.GRAY = ""


SIGNAL_COLOR = {
    "BUY": C.GREEN,
    "WATCH": C.CYAN,
    "WAIT": C.YELLOW,
    "SELL": C.RED,
    "ERR": C.MAGENTA,
}

ANSI_RE = re.compile(r"\033\[[0-9;]*m")
visible_len = lambda s: len(ANSI_RE.sub("", s))


def print_table(rows):
    """rows: لیستی از (label, feats_dict, score, signal)."""
    headers = ["Symbol", "R1", "R5", "RB", "V", "Score", "Signal"]
    mark = lambda v: f"{C.GREEN}✓{C.RESET}" if v else f"{C.GRAY}✗{C.RESET}"

    table = []
    for label, feats, score, sig in rows:
        color = SIGNAL_COLOR.get(sig, C.RESET)
        table.append([
            f"{C.BOLD}{label}{C.RESET}",
            mark(feats["R1"]),
            mark(feats["R5"]),
            mark(feats["RB"]),
            mark(feats["V"]),
            f"{score}/4",
            f"{color}{C.BOLD}{sig}{C.RESET}",
        ])

    widths = [visible_len(h) for h in headers]
    for row in table:
        for i, c in enumerate(row):
            widths[i] = max(widths[i], visible_len(c))

    def line(l, m, r):
        return l + m.join("─" * (w + 2) for w in widths) + r

    def row_str(cells, header=False):
        parts = []
        for i, cell in enumerate(cells):
            pad = widths[i] - visible_len(cell)
            s = f" {cell}{' ' * pad} "
            if header:
                s = f"{C.BOLD}{s}{C.RESET}"
            parts.append(s)
        return "│" + "│".join(parts) + "│"

    print(line("╭", "┬", "╮"))
    print(row_str(headers, header=True))
    print(line("├", "┼", "┤"))
    for row in table:
        print(row_str(row))
    print(line("╰", "┴", "╯"))


print("✅ تابع print_table آماده است")

✅ تابع print_table آماده است


In [15]:
# ---- مرحله ۱: کل بازار ----
print("📡 دریافت لیست بازار...")
market = get_market_overview()
print(f"✅ تعداد نمادهای فعال: {len(market)}")

# ---- مرحله ۲: فیلتر درشت ----
short = shortlist(market, top_n=15)
print(f"\n🔎 کاندیداها برای بررسی عمیق: {len(short)}")
print(f"   {[c['symbol'] for c in short]}\n")

# ---- مرحله ۳: امتیازدهی عمیق ----
print("🧮 در حال محاسبه فیچرها...")
rows = []
for i, c in enumerate(short, 1):
    try:
        data = load(c["symbol"])
        feats, score, sig = evaluate(data)
        rows.append((c["symbol"], feats, score, sig))
        print(f"   [{i:2d}/{len(short)}] {c['symbol']:15s} → {score}/4")
    except Exception as e:
        print(f"   [{i:2d}/{len(short)}] {c['symbol']:15s} → خطا: {e}")

# ---- مرحله ۴: مرتب و top 5 ----
rows.sort(key=lambda x: x[2], reverse=True)
top5 = rows[:5]

print(f"\n🏆 ۵ نماد برتر:\n")
print_table(top5)

📡 دریافت لیست بازار...
✅ تعداد نمادهای فعال: 0

🔎 کاندیداها برای بررسی عمیق: 0
   []

🧮 در حال محاسبه فیچرها...

🏆 ۵ نماد برتر:

╭────────┬────┬────┬────┬───┬───────┬────────╮
│ Symbol │ R1 │ R5 │ RB │ V │ Score │ Signal │
├────────┼────┼────┼────┼───┼───────┼────────┤
╰────────┴────┴────┴────┴───┴───────┴────────╯


#### ✅ پروژه کامل شد

##### خلاصه
- ۴ فیچر: بازده ۱ روزه، بازده ۵ روزه، حقیقی/حقوقی، حجم
- ۴ سطح سیگنال: BUY / WATCH / WAIT / SELL
- فیلتر دو مرحله‌ای: MarketWatch (سریع) → load (دقیق)

##### هشدار
این پروژه ابزار آموزشی است، نه توصیه سرمایه‌گذاری.